# Notebook 24d — Wu 2003 S-B3: SNPE training + held-out SBC + fault classification
# (widened lognormal prior, pooled multi-machine campaign)

## Purpose

Final step of the 2026-09-25 large-scale S-B3 campaign: train SNPE on `notebooks/22g`'s pooled
multi-machine bank, evaluate calibration against `notebooks/22f`'s dedicated held-out N=800 SBC
confirm set, and — if a seed clears SBC — run fault classification and parameter estimation against
the full 12-scenario S-B3 catalogue, using the same acceptance bar `notebooks/31e` established
(macro-F1>=0.85, all retained class F1>=0.70).

Training uses a single fixed prior (`mixed_wide_prior_4d_sb3(sigma=LOGN_SIGMA_SB3_4D_WIDE)`) — no
multi-proposal atomic-loss correction needed, unlike `notebooks/24c`, since every pooled row shares
the same true generating distribution.

**Not executed** — depends on `notebooks/22g`'s pooled bank and `notebooks/22f`'s confirm set both
existing on disk first.

## Output files

- `sbi-logs/wu2003_posterior_sb3_widelognormal.pkl`
- `results/24d_sb3_widelognormal_seedsweep_confirm_sbc_n800.csv`
- `results/24d_sb3_widelognormal_classification_metrics.csv`


## Section 1 — Imports, paths, load pooled bank + held-out confirm set


In [ ]:
import pathlib, pickle, sys, time, importlib, glob
import numpy as np
import pandas as pd
import torch
from scipy import stats as scipy_stats

cwd = pathlib.Path.cwd().resolve()
ROOT = cwd if (cwd / 'src').exists() else cwd.parent
sys.path.insert(0, str(ROOT / 'src'))

from sbi.inference import SNPE
from sbi.neural_nets import posterior_nn

import cstr_sbi.recycle.sb3 as sb3
sb3 = importlib.reload(sb3)
from cstr_sbi.recycle.sb3 import (
    LOGN_SIGMA_SB3_4D_WIDE, SB3_PARAM_NAMES, FAULT_UNITS_SB3,
    mixed_wide_prior_4d_sb3, list_closed_loop_sb3, deterministic_window_sb3,
    noisy_replicates_sb3, nominal_warm_start_sb3, fault_unit_sb3_truth, classify_fault_sb3,
)
from cstr_sbi.recycle.summaries import compute_summaries, N_SUMMARIES_SB

DATA = ROOT / 'data'
RESULTS = ROOT / 'results'; RESULTS.mkdir(exist_ok=True)
SBI_LOGS = ROOT / 'sbi-logs'; SBI_LOGS.mkdir(exist_ok=True)

NOISE_PCT = 0.003
T_FINAL_H = 4.0
N_SAVE = 240

# Update this glob if notebooks/22g produced a pooled bank with a different total row count.
pooled_candidates = sorted(DATA.glob('wu2003_sbi_train_sb3_widelognormal_pooled_n*.npz'))
assert pooled_candidates, 'No pooled bank found -- run notebooks/22e (per machine) then notebooks/22g first.'
POOLED_PATH = pooled_candidates[-1]

CONFIRM_PATH = RESULTS / '22f_sb3_widelognormal_sbc_confirm_n800.npz'
assert CONFIRM_PATH.exists(), f'{CONFIRM_PATH.name} not found -- run notebooks/22f first.'

pooled = np.load(POOLED_PATH)
theta_train = pooled['thetas4']
summary_train = pooled['summaries']

confirm = np.load(CONFIRM_PATH)
theta_confirm = confirm['thetas4']
summary_confirm = confirm['summaries']

prior = mixed_wide_prior_4d_sb3(uniform_idx=(0, 1), sigma=LOGN_SIGMA_SB3_4D_WIDE)
y0_sb3 = nominal_warm_start_sb3()

print(f'POOLED_PATH={POOLED_PATH.name}: {theta_train.shape[0]} training rows')
print(f'CONFIRM_PATH={CONFIRM_PATH.name}: {theta_confirm.shape[0]} held-out confirm rows')


## Section 2 — Seed-sweep SNPE training

SNPE training itself is cheap (~15s/seed per `notebooks/31c`'s established practice) regardless of upstream bank-generation cost, so a several-seed sweep is essentially free compared to the data-generation campaign.


In [ ]:
N_SEEDS = 10

def train_seed(seed, hidden=60, num_transforms=3):
    # hidden=60/num_transforms=3 and z_score_x='none' are this project's established S-B3/S-C
    # settings -- see feedback_network_sizing (60/3 validated, bigger AND smaller both hurt) and
    # the z_score_x='none' fix documented in HANDOFF.md.
    torch.manual_seed(seed)
    net_builder = posterior_nn(
        model='nsf', hidden_features=hidden, num_transforms=num_transforms, z_score_x='none',
    )
    inference = SNPE(prior=prior, density_estimator=net_builder)
    inference.append_simulations(
        torch.as_tensor(theta_train, dtype=torch.float32),
        torch.as_tensor(summary_train, dtype=torch.float32),
    )
    density_estimator = inference.train()
    return inference.build_posterior(density_estimator)


def run_sbc(posterior, n_post=200, seed=0):
    # Standard rank-based SBC: for each held-out confirm draw, sample the posterior at its
    # observation, compute each parameter's rank among the posterior samples, then KS-test the
    # rank fractions against Uniform(0,1). Same methodology as notebooks/24b/31c.
    n = theta_confirm.shape[0]
    n_dims = theta_confirm.shape[1]
    rank_frac = np.zeros((n, n_dims), dtype=np.float32)
    for i in range(n):
        x_o = torch.as_tensor(summary_confirm[i:i + 1], dtype=torch.float32)
        samples = posterior.sample((n_post,), x=x_o, show_progress_bars=False).numpy()
        for d in range(n_dims):
            rank_frac[i, d] = np.mean(samples[:, d] < theta_confirm[i, d])
    ks_pvalues = [scipy_stats.kstest(rank_frac[:, d], 'uniform').pvalue for d in range(n_dims)]
    return rank_frac, ks_pvalues


posteriors = {}
sweep_rows = []
t0 = time.time()
for seed in range(N_SEEDS):
    posterior = train_seed(seed)
    posteriors[seed] = posterior
    rank_frac, ks_pvalues = run_sbc(posterior, n_post=200, seed=seed)
    row = {'seed': seed, 'min_ks_pvalue': float(min(ks_pvalues))}
    for name, p in zip(SB3_PARAM_NAMES, ks_pvalues):
        row[f'{name}_ks_pvalue'] = float(p)
    sweep_rows.append(row)
    print(f"seed={seed}: min_ks_p={row['min_ks_pvalue']:.4g}  " +
          ', '.join(f'{n}={p:.4g}' for n, p in zip(SB3_PARAM_NAMES, ks_pvalues)))

print(f'seed sweep done in {time.time() - t0:.0f}s')
sweep_df = pd.DataFrame(sweep_rows)
sweep_df.to_csv(RESULTS / '24d_sb3_widelognormal_seedsweep_confirm_sbc_n800.csv', index=False)
best_seed = int(sweep_df.loc[sweep_df['min_ks_pvalue'].idxmax(), 'seed'])
print(f'best seed by min_ks_pvalue: {best_seed}')


## Section 3 — Save the selected posterior


In [ ]:
best_posterior = posteriors[best_seed]
POSTERIOR_PATH = SBI_LOGS / 'wu2003_posterior_sb3_widelognormal.pkl'
with open(POSTERIOR_PATH, 'wb') as f:
    pickle.dump({
        'posterior': best_posterior,
        'seed': best_seed,
        'structure': 'S-B3',
        'prior_tag': 'widelognormal_v1',
        'bank': POOLED_PATH.name,
        'confirm_set': CONFIRM_PATH.name,
    }, f)
print(f'saved {POSTERIOR_PATH.relative_to(ROOT)}')


## Section 4 — Fault classification and parameter estimation (12-scenario catalogue)

Mirrors `notebooks/31e`'s pooled-replicate classification approach: simulate each retained scenario, generate noisy replicates, sample the posterior per replicate, and classify from the pooled posterior sample across all replicates. Uses the module's default prediction threshold (`PRED_UNIT_THRESH_SB3`); re-tune per `notebooks/31c`'s pattern if this underperforms.


In [ ]:
N_CLS_REPLICATES = 30
N_POST_CLS = 500


def classify_scenario(scenario, posterior, rng):
    theta4_true = np.asarray(
        [scenario.alpha, scenario.beta_r, scenario.eta_col, scenario.xi_reb], dtype=np.float32
    )
    t_h, raw_obs = deterministic_window_sb3(scenario, y0=y0_sb3, t_final_h=T_FINAL_H, n_save=N_SAVE)
    noisy = noisy_replicates_sb3(
        np.asarray(raw_obs), n_replicates=N_CLS_REPLICATES, rng=rng, noise_pct=NOISE_PCT,
    )
    preds, all_samples = [], []
    for rep in noisy:
        summary = compute_summaries(rep, 'S-B', np.asarray(t_h))
        x_o = torch.as_tensor(summary[None, :], dtype=torch.float32)
        samples = posterior.sample((N_POST_CLS,), x=x_o, show_progress_bars=False).numpy()
        all_samples.append(samples)
        preds.append(classify_fault_sb3(samples)['class'])
    pooled_samples = np.concatenate(all_samples, axis=0)
    pooled = classify_fault_sb3(pooled_samples)
    true_unit = fault_unit_sb3_truth(theta4_true)
    row = {
        'scenario': scenario.name, 'true_unit': true_unit, 'pooled_pred': pooled['class'],
        'replicate_accuracy': float(np.mean([p == true_unit for p in preds])),
    }
    for i, name in enumerate(SB3_PARAM_NAMES):
        row[f'{name}_true'] = float(theta4_true[i])
        row[f'{name}_post_mean'] = float(pooled['means'][name])
    return row


rng_cls = np.random.default_rng(20260925)
scenario_rows = [classify_scenario(sc, best_posterior, rng_cls) for sc in list_closed_loop_sb3()]
results_df = pd.DataFrame(scenario_rows)
results_df.to_csv(RESULTS / '24d_sb3_widelognormal_classification_metrics.csv', index=False)
print(results_df.to_string(index=False))


## Section 5 — Confusion matrix and macro-F1


In [ ]:
def compute_metrics(df, classes=FAULT_UNITS_SB3):
    y_true = df['true_unit'].tolist()
    y_pred = df['pooled_pred'].tolist()
    idx = {c: i for i, c in enumerate(classes)}
    cm = np.zeros((len(classes), len(classes)), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[idx[t], idx[p]] += 1
    f1 = {}
    for c in classes:
        tp = cm[idx[c], idx[c]]
        fp = cm[:, idx[c]].sum() - tp
        fn = cm[idx[c], :].sum() - tp
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec = tp / (tp + fn) if (tp + fn) else 0.0
        f1[c] = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return {
        'confusion_matrix': cm, 'per_class_f1': f1,
        'macro_f1': float(np.mean(list(f1.values()))),
        'accuracy': float(np.trace(cm) / cm.sum()),
    }


metrics = compute_metrics(results_df)
print('accuracy', metrics['accuracy'], 'macro_f1', metrics['macro_f1'])
print(metrics['per_class_f1'])
print(metrics['confusion_matrix'])


## Section 6 — Final assessment

Loads `notebooks/31e`'s own saved classification CSV for a like-for-like baseline comparison (computed from the file, not hardcoded, so this stays correct if that file is ever regenerated).


In [ ]:
print('=== Final assessment ===')
print(f'Selected seed: {best_seed}')
sweep_row = sweep_df[sweep_df['seed'] == best_seed].iloc[0]
for name in SB3_PARAM_NAMES:
    print(f"  {name}: KS p={sweep_row[f'{name}_ks_pvalue']:.4g}")
print()

criteria = [
    ('all 4 parameters pass SBC (p >= 0.05)',
     all(sweep_row[f'{n}_ks_pvalue'] >= 0.05 for n in SB3_PARAM_NAMES)),
    ('macro-F1 >= 0.85', metrics['macro_f1'] >= 0.85),
    ('all retained class F1 >= 0.70', all(v >= 0.70 for v in metrics['per_class_f1'].values())),
]
for name, passed in criteria:
    print(f"{'PASS' if passed else 'FAIL'} | {name}")
print()

baseline_path = RESULTS / '31e_fault_classification_metrics.csv'
if baseline_path.exists():
    baseline_df = pd.read_csv(baseline_path)
    baseline_scenario_acc = float(baseline_df['scenario_accuracy'].mean())
    print(f'Baseline (notebooks/31e, narrow-lognormal prior, seed 5): '
          f'mean scenario_accuracy={baseline_scenario_acc:.3f}')
    print("Baseline SBC caveat (nb31e's own printed assessment): alpha p=0.0006, beta_r p=0.038, "
          "xi_reb p=0.0009 fail formal calibration; only eta_col p=0.531 passes.")
    run_scenario_acc = float(results_df['replicate_accuracy'].mean())
    print(f'This run: mean scenario_accuracy={run_scenario_acc:.3f}, '
          f"macro-F1={metrics['macro_f1']:.3f}")
else:
    print(f'{baseline_path.name} not found -- skipping baseline comparison.')
print()

if not criteria[0][1]:
    print('SBC did not fully clear for all 4 parameters. Do NOT promote this posterior as fully')
    print('calibrated. Before another training/architecture sweep, check for a shrinkage-toward-')
    print('mode signature or a genuine residual confound (see this project\'s')
    print('finding_sc_delta_dimension_hypothesis / project_sc_alpha_xireb_shrinkage memory) --')
    print('history here shows blind hyperparameter sweeps do not fix this failure mode.')
